# Co-Training with Llama Pseudo-Labels

Full experiment run: **2 tasks x 3 modalities x 4 budgets x 3 seeds = 72 experiments**

| Modality | Model |
| --- | --- |
| text_only | BERTweet (`vinai/bertweet-base`) |
| image_only | CLIP ViT (`openai/clip-vit-base-patch32`) |
| text_image | BERTweet + CLIP ViT (late fusion) |

Results saved under `results/cotrain/lg-cotrain/llama-3.2-11b/{run_id}/...`  
Completed experiments are automatically skipped on re-run.

**Prerequisite:** Run `01_zeroshot_llama3.2-11B.ipynb` first, then generate pseudo-labels:
```bash
python scripts/create_pseudo_labels.py --model llama-3.2-11b
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import time
import pandas as pd
from lg_cotrain.run_all import run_all_experiments, format_summary_table

In [2]:
# ---- Configuration ----
RUN_ID = "run-1"  # Change to run-2, run-3 etc. for different settings
NUM_GPUS = 2      # Parallel experiments across GPUs (1 = sequential)

PARAMS = dict(
    pseudo_label_source="llama-3.2-11b",
    run_id=RUN_ID,
    # Paper defaults
    weight_gen_epochs=7,
    cotrain_epochs=10,
    finetune_max_epochs=100,
    finetune_patience=5,
    batch_size=32,
    lr=2e-5,
)

DATA_ROOT = os.path.abspath("../data")
RESULTS_ROOT = os.path.abspath("../results")

TASKS = ["informative", "humanitarian"]
MODALITIES = ["text_only", "image_only", "text_image"]
BUDGETS = [5, 10, 25, 50]
SEED_SETS = [1, 2, 3]

total = len(TASKS) * len(MODALITIES) * len(BUDGETS) * len(SEED_SETS)
print(f"Run ID:       {RUN_ID}")
print(f"GPUs:         {NUM_GPUS}")
print(f"Data root:    {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Output path:  results/cotrain/lg-cotrain/llama-3.2-11b/{RUN_ID}/{{task}}/{{modality}}/...")
print(f"Total experiments: {total}")

Run ID:       run-1
GPUs:         2
Data root:    D:\Workspace\Cotrain_CrisisMMD\data
Results root: D:\Workspace\Cotrain_CrisisMMD\results
Output path:  results/cotrain/lg-cotrain/llama-3.2-11b/run-1/{task}/{modality}/...
Total experiments: 72


In [ ]:
import gc, torch

all_summaries = {}  # (task, modality) -> results list

def run_batch(task, modality):
    """Run all 12 experiments for one (task, modality) and print summary."""
    print(f"\n{'='*70}")
    print(f"  {task} / {modality}  \u2014  {len(BUDGETS)}x{len(SEED_SETS)} = {len(BUDGETS)*len(SEED_SETS)} experiments")
    print(f"{'='*70}")
    
    n_total = len(BUDGETS) * len(SEED_SETS)
    progress = {"done": 0, "start": time.time()}

    def on_done(task, modality, budget, seed_set, status):
        progress["done"] += 1
        elapsed = time.time() - progress["start"]
        n = progress["done"]
        avg = elapsed / n
        eta = avg * (n_total - n)
        print(f"  [{n}/{n_total}] budget={budget}, seed={seed_set} -- {status} "
              f"({elapsed/60:.1f}min elapsed, ~{eta/60:.1f}min remaining)")

    start = time.time()
    results = run_all_experiments(
        task, modality,
        budgets=BUDGETS,
        seed_sets=SEED_SETS,
        num_gpus=NUM_GPUS,
        data_root=DATA_ROOT,
        results_root=RESULTS_ROOT,
        _on_experiment_done=on_done,
        **PARAMS,
    )
    elapsed = time.time() - start
    
    all_summaries[(task, modality)] = results
    
    print(f"\nCompleted {task}/{modality} in {elapsed/60:.1f}min")
    print(format_summary_table(results, task, modality,
                               budgets=BUDGETS, seed_sets=SEED_SETS))
    
    # Free GPU memory before next batch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

## Informative Task

In [ ]:
run_batch("informative", "text_only")


  informative / text_only  --  4x3 = 12 experiments
budget=5, seed=1 -- SKIPPED (exists)
  [1/12] budget=5, seed=1 -- skipped (0s elapsed, ~0s remaining)
budget=5, seed=2 -- SKIPPED (exists)
  [2/12] budget=5, seed=2 -- skipped (0s elapsed, ~0s remaining)
budget=5, seed=3 -- SKIPPED (exists)
  [3/12] budget=5, seed=3 -- skipped (0s elapsed, ~0s remaining)
budget=10, seed=1 -- SKIPPED (exists)
  [4/12] budget=10, seed=1 -- skipped (0s elapsed, ~0s remaining)
budget=10, seed=2 -- SKIPPED (exists)
  [5/12] budget=10, seed=2 -- skipped (0s elapsed, ~0s remaining)
Running 7 experiments in parallel across 2 GPUs...
  [6/12] budget=25, seed=1 -- done (1343s elapsed, ~1343s remaining)
  [7/12] budget=10, seed=3 -- done (1480s elapsed, ~1057s remaining)


In [ ]:
run_batch("informative", "image_only")

In [ ]:
run_batch("informative", "text_image")

## Humanitarian Task

In [ ]:
run_batch("humanitarian", "text_only")

In [ ]:
run_batch("humanitarian", "image_only")

In [ ]:
run_batch("humanitarian", "text_image")

## Cross-Modality Summary

In [ ]:
import numpy as np

rows = []
for (task, modality), results in all_summaries.items():
    valid = [r for r in results if r is not None]
    if not valid:
        rows.append({"Task": task, "Modality": modality,
                     "N": 0, "Mean F1": "-", "Std F1": "-",
                     "Mean Err%": "-", "Mean ECE": "-"})
        continue
    f1s = [r["test_macro_f1"] for r in valid]
    errs = [r["test_error_rate"] for r in valid]
    eces = [r["test_ece"] for r in valid]
    rows.append({
        "Task": task,
        "Modality": modality,
        "N": len(valid),
        "Mean F1": f"{np.mean(f1s):.4f}",
        "Std F1": f"{np.std(f1s):.4f}",
        "Mean Err%": f"{np.mean(errs):.2f}",
        "Mean ECE": f"{np.mean(eces):.4f}",
    })

df = pd.DataFrame(rows)
print(f"Cross-Modality Summary ({RUN_ID})")
print(f"Averaged over {len(BUDGETS)} budgets x {len(SEED_SETS)} seeds = {len(BUDGETS)*len(SEED_SETS)} experiments each")
print("=" * 70)
df

## Cleanup

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory freed. Peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("No GPU available (ran on CPU)")

n_success = sum(1 for results in all_summaries.values() for r in results if r is not None)
print(f"\nDone. {n_success}/{total} experiments completed successfully.")